# Solace Browser Deployment Notebook 04 — Android Play Store

**Version:** 1.0.0 | **Auth:** 65537 | **Rung Target:** 65537

NORTHSTAR: build a signed release APK, upload to Google Play via the Developer API, promote through testing tracks to production, and seal evidence with SHA-256 hash chain.

## Prerequisites

- Android SDK installed with `ANDROID_HOME` set
- Release keystore at `$SOLACE_KEYSTORE_PATH` (env var)
- Google Play Developer API service account JSON at `$GOOGLE_PLAY_SERVICE_ACCOUNT_JSON`
- `google-api-python-client` installed (`pip install google-api-python-client google-auth`)
- `VERSION` file in project root
- Gradle wrapper (`gradlew`) in `android/` directory

In [ ]:
"""PREFLIGHT — verify APK build prerequisites, signing key, and AndroidManifest."""
import os
import subprocess
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path('/home/phuc/projects/solace-browser')
ANDROID_DIR = PROJECT_ROOT / 'android'
VERSION = (PROJECT_ROOT / 'VERSION').read_text(encoding='utf-8').strip()
TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
EVIDENCE_DIR = PROJECT_ROOT / 'scratch' / 'play-store-deploy' / TIMESTAMP
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

# --- Verify Android SDK ---
ANDROID_HOME = os.environ.get('ANDROID_HOME', '')
assert ANDROID_HOME and Path(ANDROID_HOME).is_dir(), \
    f'ANDROID_HOME not set or not a directory: {ANDROID_HOME!r}'

# --- Verify Gradle wrapper ---
GRADLEW = ANDROID_DIR / 'gradlew'
assert GRADLEW.is_file(), f'gradlew not found at {GRADLEW}'
assert os.access(GRADLEW, os.X_OK), f'gradlew not executable: {GRADLEW}'

# --- Verify signing keystore ---
KEYSTORE_PATH = os.environ.get('SOLACE_KEYSTORE_PATH', '')
assert KEYSTORE_PATH and Path(KEYSTORE_PATH).is_file(), \
    f'SOLACE_KEYSTORE_PATH not set or file missing: {KEYSTORE_PATH!r}'
KEYSTORE_PASS = os.environ.get('SOLACE_KEYSTORE_PASSWORD', '')
assert KEYSTORE_PASS, 'SOLACE_KEYSTORE_PASSWORD env var not set'
KEY_ALIAS = os.environ.get('SOLACE_KEY_ALIAS', 'solace-browser')
KEY_PASSWORD = os.environ.get('SOLACE_KEY_PASSWORD', KEYSTORE_PASS)

# --- Verify keystore is valid via keytool ---
keytool_result = subprocess.run(
    ['keytool', '-list', '-keystore', KEYSTORE_PATH,
     '-storepass', KEYSTORE_PASS, '-alias', KEY_ALIAS],
    capture_output=True, text=True, timeout=30
)
assert keytool_result.returncode == 0, \
    f'keytool verification failed: {keytool_result.stderr}'
print(f'Keystore alias {KEY_ALIAS!r} verified in {KEYSTORE_PATH}')

# --- Verify AndroidManifest.xml ---
MANIFEST = ANDROID_DIR / 'app' / 'src' / 'main' / 'AndroidManifest.xml'
assert MANIFEST.is_file(), f'AndroidManifest.xml not found at {MANIFEST}'
manifest_text = MANIFEST.read_text(encoding='utf-8')
assert 'com.solaceagi.browser' in manifest_text, \
    'AndroidManifest.xml missing expected package name com.solaceagi.browser'

# --- Verify build.gradle has correct versionName ---
BUILD_GRADLE = ANDROID_DIR / 'app' / 'build.gradle'
assert BUILD_GRADLE.is_file(), f'build.gradle not found at {BUILD_GRADLE}'
gradle_text = BUILD_GRADLE.read_text(encoding='utf-8')
assert VERSION in gradle_text, \
    f'build.gradle does not contain version {VERSION}'

# --- Verify Google Play service account ---
SA_JSON_PATH = os.environ.get('GOOGLE_PLAY_SERVICE_ACCOUNT_JSON', '')
assert SA_JSON_PATH and Path(SA_JSON_PATH).is_file(), \
    f'GOOGLE_PLAY_SERVICE_ACCOUNT_JSON not set or missing: {SA_JSON_PATH!r}'
sa_data = json.loads(Path(SA_JSON_PATH).read_text(encoding='utf-8'))
assert sa_data.get('type') == 'service_account', \
    'Service account JSON does not have type=service_account'
print(f'Service account: {sa_data.get("client_email", "UNKNOWN")}')

preflight = {
    'status': 'ok',
    'timestamp': TIMESTAMP,
    'version': VERSION,
    'android_home': ANDROID_HOME,
    'keystore_alias': KEY_ALIAS,
    'manifest_package': 'com.solaceagi.browser',
    'service_account': sa_data.get('client_email', 'UNKNOWN'),
    'evidence_dir': str(EVIDENCE_DIR),
}
(EVIDENCE_DIR / 'preflight.json').write_text(
    json.dumps(preflight, indent=2), encoding='utf-8'
)
print(json.dumps(preflight, indent=2))

In [ ]:
"""BEFORE SNAPSHOT — check current Play Store listing status."""
from google.oauth2 import service_account
from googleapiclient.discovery import build as google_build

PACKAGE_NAME = 'com.solaceagi.browser'
SCOPES = ['https://www.googleapis.com/auth/androidpublisher']

credentials = service_account.Credentials.from_service_account_file(
    SA_JSON_PATH, scopes=SCOPES
)
play_service = google_build('androidpublisher', 'v3', credentials=credentials)

# --- Get current tracks ---
edit_request = play_service.edits().insert(
    packageName=PACKAGE_NAME, body={}
)
edit = edit_request.execute()
edit_id = edit['id']

tracks_response = play_service.edits().tracks().list(
    packageName=PACKAGE_NAME, editId=edit_id
).execute()

before_snapshot = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'package': PACKAGE_NAME,
    'tracks': [],
}

for track in tracks_response.get('tracks', []):
    track_info = {
        'track': track['track'],
        'releases': []
    }
    for release in track.get('releases', []):
        track_info['releases'].append({
            'status': release.get('status', 'UNKNOWN'),
            'versionCodes': release.get('versionCodes', []),
            'name': release.get('name', ''),
        })
    before_snapshot['tracks'].append(track_info)

# Discard the edit (read-only snapshot)
play_service.edits().delete(
    packageName=PACKAGE_NAME, editId=edit_id
).execute()

(EVIDENCE_DIR / 'before-snapshot.json').write_text(
    json.dumps(before_snapshot, indent=2), encoding='utf-8'
)
print(json.dumps(before_snapshot, indent=2))

In [ ]:
"""BUILD — run gradlew assembleRelease and sign APK with keystore."""
import shutil

# --- Clean previous build ---
clean_result = subprocess.run(
    [str(GRADLEW), 'clean'],
    cwd=str(ANDROID_DIR),
    capture_output=True, text=True, timeout=300
)
assert clean_result.returncode == 0, \
    f'gradle clean failed: {clean_result.stderr}'
print('gradle clean: OK')

# --- Build release APK ---
build_env = os.environ.copy()
build_env.update({
    'SOLACE_KEYSTORE_PATH': KEYSTORE_PATH,
    'SOLACE_KEYSTORE_PASSWORD': KEYSTORE_PASS,
    'SOLACE_KEY_ALIAS': KEY_ALIAS,
    'SOLACE_KEY_PASSWORD': KEY_PASSWORD,
})

build_result = subprocess.run(
    [str(GRADLEW), 'assembleRelease',
     f'-Pandroid.injected.signing.store.file={KEYSTORE_PATH}',
     f'-Pandroid.injected.signing.store.password={KEYSTORE_PASS}',
     f'-Pandroid.injected.signing.key.alias={KEY_ALIAS}',
     f'-Pandroid.injected.signing.key.password={KEY_PASSWORD}'],
    cwd=str(ANDROID_DIR),
    env=build_env,
    capture_output=True, text=True, timeout=600
)

# Save build log regardless of outcome
build_log_path = EVIDENCE_DIR / 'gradle-build.log'
build_log_path.write_text(
    f'STDOUT:\n{build_result.stdout}\n\nSTDERR:\n{build_result.stderr}',
    encoding='utf-8'
)
assert build_result.returncode == 0, \
    f'assembleRelease failed (log: {build_log_path}): {build_result.stderr[-500:]}'
print('assembleRelease: OK')

# --- Locate signed APK ---
APK_DIR = ANDROID_DIR / 'app' / 'build' / 'outputs' / 'apk' / 'release'
apk_candidates = list(APK_DIR.glob('*.apk'))
assert len(apk_candidates) > 0, f'No APK found in {APK_DIR}'
# Prefer the unsigned-free variant
signed_apks = [a for a in apk_candidates if 'unsigned' not in a.name]
APK_PATH = signed_apks[0] if signed_apks else apk_candidates[0]

# --- Verify APK signature ---
verify_result = subprocess.run(
    [os.path.join(ANDROID_HOME, 'build-tools', '34.0.0', 'apksigner'),
     'verify', '--verbose', str(APK_PATH)],
    capture_output=True, text=True, timeout=60
)
assert verify_result.returncode == 0, \
    f'APK signature verification failed: {verify_result.stderr}'
print(f'APK signature verified: {APK_PATH.name}')

# --- Compute APK hash ---
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

APK_HASH = sha256_file(APK_PATH)
APK_SIZE = APK_PATH.stat().st_size

# --- Copy APK to evidence ---
evidence_apk = EVIDENCE_DIR / APK_PATH.name
shutil.copy2(APK_PATH, evidence_apk)

build_evidence = {
    'status': 'ok',
    'apk_path': str(APK_PATH),
    'apk_name': APK_PATH.name,
    'apk_sha256': APK_HASH,
    'apk_size_bytes': APK_SIZE,
    'apk_size_mb': round(APK_SIZE / (1024 * 1024), 2),
    'version': VERSION,
    'signature_verified': True,
}
(EVIDENCE_DIR / 'build-evidence.json').write_text(
    json.dumps(build_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(build_evidence, indent=2))

In [ ]:
"""UPLOAD — upload signed APK to Google Play internal testing track."""
from googleapiclient.http import MediaFileUpload

# Create a new edit
edit = play_service.edits().insert(
    packageName=PACKAGE_NAME, body={}
).execute()
edit_id = edit['id']
print(f'Created edit: {edit_id}')

# Upload APK
media = MediaFileUpload(str(APK_PATH), mimetype='application/vnd.android.package-archive')
upload_response = play_service.edits().apks().upload(
    packageName=PACKAGE_NAME,
    editId=edit_id,
    media_body=media
).execute()

version_code = upload_response['versionCode']
print(f'Uploaded APK with versionCode: {version_code}')

# Assign to internal testing track
track_body = {
    'track': 'internal',
    'releases': [{
        'versionCodes': [str(version_code)],
        'name': f'v{VERSION}',
        'status': 'completed',
        'releaseNotes': [{
            'language': 'en-US',
            'text': f'Solace Browser v{VERSION} — internal testing build'
        }]
    }]
}
track_response = play_service.edits().tracks().update(
    packageName=PACKAGE_NAME,
    editId=edit_id,
    track='internal',
    body=track_body
).execute()
print(f'Assigned to internal track: {json.dumps(track_response, indent=2)}')

# Commit the edit
commit_response = play_service.edits().commit(
    packageName=PACKAGE_NAME, editId=edit_id
).execute()
print(f'Edit committed: {commit_response}')

upload_evidence = {
    'status': 'ok',
    'edit_id': edit_id,
    'version_code': version_code,
    'version_name': f'v{VERSION}',
    'track': 'internal',
    'timestamp': datetime.now(timezone.utc).isoformat(),
}
(EVIDENCE_DIR / 'upload-evidence.json').write_text(
    json.dumps(upload_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(upload_evidence, indent=2))

In [ ]:
"""PROMOTE — promote from internal -> closed testing -> open testing -> production.

Each promotion is a separate edit+commit to maintain evidence chain.
Operator MUST verify test results between promotions.
"""

PROMOTION_LADDER = [
    ('internal', 'alpha'),      # internal -> closed testing
    ('alpha', 'beta'),          # closed testing -> open testing
    ('beta', 'production'),     # open testing -> production
]

# --- Set TARGET_TRACK to control how far to promote ---
# Options: 'alpha', 'beta', 'production'
# Default: promote only to closed testing (alpha)
TARGET_TRACK = os.environ.get('SOLACE_PROMOTE_TARGET', 'alpha')
print(f'Promotion target: {TARGET_TRACK}')

promotion_log = []

for from_track, to_track in PROMOTION_LADDER:
    print(f'\n--- Promoting {from_track} -> {to_track} ---')

    edit = play_service.edits().insert(
        packageName=PACKAGE_NAME, body={}
    ).execute()
    eid = edit['id']

    # Read source track to get current version codes
    source_track = play_service.edits().tracks().get(
        packageName=PACKAGE_NAME, editId=eid, track=from_track
    ).execute()

    current_releases = source_track.get('releases', [])
    completed = [r for r in current_releases if r.get('status') == 'completed']
    assert len(completed) > 0, \
        f'No completed release on {from_track} track to promote'

    latest_release = completed[0]
    codes = latest_release['versionCodes']

    # Assign to destination track
    dest_body = {
        'track': to_track,
        'releases': [{
            'versionCodes': codes,
            'name': f'v{VERSION}',
            'status': 'completed',
            'releaseNotes': [{
                'language': 'en-US',
                'text': f'Solace Browser v{VERSION} — promoted from {from_track}'
            }]
        }]
    }

    play_service.edits().tracks().update(
        packageName=PACKAGE_NAME,
        editId=eid,
        track=to_track,
        body=dest_body
    ).execute()

    play_service.edits().commit(
        packageName=PACKAGE_NAME, editId=eid
    ).execute()

    step = {
        'from': from_track,
        'to': to_track,
        'version_codes': codes,
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }
    promotion_log.append(step)
    print(f'Promoted: {json.dumps(step, indent=2)}')

    if to_track == TARGET_TRACK:
        print(f'\nReached target track: {TARGET_TRACK}. Stopping.')
        break

(EVIDENCE_DIR / 'promotion-log.json').write_text(
    json.dumps(promotion_log, indent=2), encoding='utf-8'
)
print(f'\nPromotion log saved: {len(promotion_log)} steps')

In [ ]:
"""VERIFY — check Play Store listing is live and verify APK hash."""
import urllib.request

# --- Verify via Play Developer API ---
edit = play_service.edits().insert(
    packageName=PACKAGE_NAME, body={}
).execute()
edit_id = edit['id']

after_tracks = play_service.edits().tracks().list(
    packageName=PACKAGE_NAME, editId=edit_id
).execute()

play_service.edits().delete(
    packageName=PACKAGE_NAME, editId=edit_id
).execute()

after_snapshot = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'package': PACKAGE_NAME,
    'tracks': [],
}
target_found = False
for track in after_tracks.get('tracks', []):
    track_info = {
        'track': track['track'],
        'releases': []
    }
    for release in track.get('releases', []):
        track_info['releases'].append({
            'status': release.get('status', 'UNKNOWN'),
            'versionCodes': release.get('versionCodes', []),
            'name': release.get('name', ''),
        })
        if track['track'] == TARGET_TRACK and release.get('status') == 'completed':
            if str(version_code) in [str(c) for c in release.get('versionCodes', [])]:
                target_found = True
    after_snapshot['tracks'].append(track_info)

assert target_found, \
    f'versionCode {version_code} not found as completed on {TARGET_TRACK} track'
print(f'VERIFIED: versionCode {version_code} is live on {TARGET_TRACK} track')

# --- Verify Play Store public listing (production only) ---
if TARGET_TRACK == 'production':
    play_url = f'https://play.google.com/store/apps/details?id={PACKAGE_NAME}'
    req = urllib.request.Request(play_url, method='HEAD')
    try:
        resp = urllib.request.urlopen(req, timeout=15)
        listing_status = resp.getcode()
    except urllib.error.HTTPError as e:
        listing_status = e.code
    print(f'Play Store listing HTTP status: {listing_status}')
    assert listing_status == 200, \
        f'Play Store listing not accessible (HTTP {listing_status})'
else:
    listing_status = 'N/A (not production track)'
    print(f'Skipping public listing check: {TARGET_TRACK} is not production')

# --- Verify APK hash matches build evidence ---
evidence_apk_hash = sha256_file(evidence_apk)
assert evidence_apk_hash == APK_HASH, \
    f'Evidence APK hash mismatch: {evidence_apk_hash} != {APK_HASH}'
print(f'APK hash verified: {APK_HASH}')

verify_evidence = {
    'status': 'ok',
    'version_code': version_code,
    'target_track': TARGET_TRACK,
    'track_verified': target_found,
    'listing_status': listing_status,
    'apk_hash_verified': True,
    'after_tracks': after_snapshot['tracks'],
    'timestamp': datetime.now(timezone.utc).isoformat(),
}
(EVIDENCE_DIR / 'after-snapshot.json').write_text(
    json.dumps(after_snapshot, indent=2), encoding='utf-8'
)
(EVIDENCE_DIR / 'verify-evidence.json').write_text(
    json.dumps(verify_evidence, indent=2), encoding='utf-8'
)
print(json.dumps(verify_evidence, indent=2))

In [ ]:
"""EVIDENCE SEAL — SHA-256 report with version, build number, APK hash.

Produces a tamper-evident evidence chain linking every artifact
from this deployment session.
"""

# --- Collect all evidence file hashes ---
evidence_files = sorted(EVIDENCE_DIR.glob('*.json'))
evidence_hashes = {}
for ef in evidence_files:
    evidence_hashes[ef.name] = sha256_file(ef)

# --- Build hash chain ---
chain_input = '\n'.join(
    f'{h}  {name}' for name, h in sorted(evidence_hashes.items())
)
chain_hash = hashlib.sha256(chain_input.encode('utf-8')).hexdigest()

# --- Git state ---
git_sha = subprocess.run(
    ['git', 'rev-parse', 'HEAD'],
    cwd=str(PROJECT_ROOT),
    capture_output=True, text=True, timeout=10
).stdout.strip()

git_dirty = subprocess.run(
    ['git', 'status', '--porcelain'],
    cwd=str(PROJECT_ROOT),
    capture_output=True, text=True, timeout=10
).stdout.strip()

seal = {
    'seal': 'SOLACE-ANDROID-DEPLOY',
    'auth': 65537,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'version': VERSION,
    'version_code': version_code,
    'package': PACKAGE_NAME,
    'target_track': TARGET_TRACK,
    'apk_name': APK_PATH.name,
    'apk_sha256': APK_HASH,
    'apk_size_bytes': APK_SIZE,
    'git_sha': git_sha,
    'git_dirty': bool(git_dirty),
    'evidence_chain': {
        'files': evidence_hashes,
        'chain_hash': chain_hash,
    },
}

seal_path = EVIDENCE_DIR / 'evidence-seal.json'
seal_path.write_text(json.dumps(seal, indent=2), encoding='utf-8')

# --- Write human-readable report ---
report_lines = [
    f'# Android Play Store Deploy Evidence — v{VERSION}',
    f'',
    f'**Timestamp:** {seal["timestamp"]}',
    f'**Auth:** {seal["auth"]}',
    f'**Package:** {PACKAGE_NAME}',
    f'**Version Name:** v{VERSION}',
    f'**Version Code:** {version_code}',
    f'**Target Track:** {TARGET_TRACK}',
    f'**Git SHA:** {git_sha}',
    f'**Git Dirty:** {bool(git_dirty)}',
    f'',
    f'## APK',
    f'- **File:** {APK_PATH.name}',
    f'- **SHA-256:** `{APK_HASH}`',
    f'- **Size:** {APK_SIZE:,} bytes ({round(APK_SIZE / (1024*1024), 2)} MB)',
    f'',
    f'## Evidence Chain',
    f'- **Chain Hash:** `{chain_hash}`',
    f'',
    f'### Evidence Files',
]
for name, h in sorted(evidence_hashes.items()):
    report_lines.append(f'- `{h}  {name}`')

report_path = EVIDENCE_DIR / 'deploy-report.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

print(f'Evidence sealed: {seal_path}')
print(f'Report written: {report_path}')
print(f'Chain hash: {chain_hash}')
print(json.dumps(seal, indent=2))